In [196]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [231]:
yerevan_historical = pd.read_csv('../database/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../database/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [232]:
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [233]:
yerevan_actual['backup_status'] = 'not sold'

In [234]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [235]:
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

In [236]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",2022-10-08 21:56:34,1.0,NaN,1851.851852,...,2.8,1.0,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513121,40.166179
18279,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",NaN,1.0,NaN,2314.814815,...,2.8,1.0,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513120,40.166179


In [237]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [238]:
df = df.dropna(subset=['district']).reset_index(drop=True)

### classifying neighborhoods

In [55]:
df2 = df.head(80592).reset_index(drop=True)

In [32]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

In [63]:
df.to_csv(r'../database/yerevan_combined_clean/sale_combined.csv', index=False)

### small amount of rows that have yerevan classfied replaced with district value 

In [206]:
df = pd.read_csv('../database/yerevan_combined_clean/sale_combined.csv')

In [150]:
for index, row in df[df['neighborhood'] == 'Yerevan'].iterrows():
    df.loc[index, 'neighborhood'] = row['district']

In [151]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [207]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [158]:
# removing neighborhoods with low counts
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [159]:
# removing neighborhoods with low counts
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [166]:
df = pd.concat([sale_historical, sale_actual], axis=0)

for some reason when scraping the data there were missing values for square-meters when price-per-meter had a value and the other way around

In [166]:
for index, row in df.iterrows():
    if np.isnan(row['price_per_meter']):
        continue
    else:
        df.loc[index, 'square_meters'] = row['price'] / row['price_per_meter']

In [167]:
for index, row in df.iterrows():
    if np.isnan(row['square_meters']):
        continue
    else:
        df.loc[index, 'price_per_meter'] = row['price'] / row['square_meters']

In [171]:
df.to_csv(r'../database/yerevan_combined_clean/sale_combined.csv', index=False)

In [14]:
df = pd.read_csv('../database/yerevan_combined_clean/sale_combined.csv')

In [67]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [17]:
sale_actual.drop(columns=[ 'sold_date'], inplace=True)

In [19]:
sale_historical = sale_historical.dropna(subset=['sold_date'])

In [24]:
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [243]:
sale_historical.to_csv(r'../database/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../database/yerevan_combined_clean/actual_not_sold.csv', index=False)

In [246]:
sale_historical.sort_values('square_meters')

,backup_status,id,price,rooms,square_meters,address,furniture,renovation,price_per_meter,floor,...,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood,sold_date
60010,sold,19017091,41000.000000,1.0,20.0,"Ֆրունզեի փողոց, Երևան",0.0,Major Renovation,2050.000000,1.0,...,NaN,44.495487,40.147234,Gortsaranain,768,Shengavit,44.495492,40.147240,Verin_Shengavit,3/14/2023
56233,sold,18790483,22460.693790,1.0,20.0,"Hrant Vardanyan Street, Yerevan",0.0,Old Renovation,1123.034689,3.0,...,1/3/2023 17:37,44.501425,40.147062,Gortsaranain,1083,Shengavit,44.501432,40.147066,Shengavit,1/27/2023
18902,sold,15599139,27500.000000,1.0,20.0,"Davidashen 4-th block, Yerevan",0.0,No Renovation,1375.000000,6.0,...,9/13/2022 1:47,44.499853,40.222106,Barekamutyun,4109,Davtashen,44.499847,40.222108,4th_Block,1/12/2023
24189,sold,17170282,29000.000000,1.0,20.0,"3-я улица Вардашена, Ереван",0.0,Euro Renovation,1450.000000,3.0,...,9/9/2022 15:17,44.542541,40.142408,Sasuntsi Davit,4436,Erebuni,44.542542,40.142398,Erebuni,1/27/2023
49124,sold,18426165,29947.591710,1.0,20.0,"Մոլդովական փողոց 29/1, Երևան",0.0,Major Renovation,1497.379586,4.0,...,11/22/2022 18:57,44.571691,40.208994,Yeritasardakan,7183,Nor_Nork,44.571683,40.208985,Nor_Nork_2nd_Microdistrict,12/6/2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55315,sold,18733062,53000.000000,NaN,NaN,"Յոհաննես Լեփսիուսի 2-րդ փողոց 1, Երևան",0.0,NaN,NaN,NaN,...,1/3/2023 17:33,44.541921,40.209442,Yeritasardakan,5296,Qanaqer_Zeytun,44.541919,40.209437,Nor_Zeytun,3/14/2023
55318,sold,18733466,29900.000000,NaN,NaN,"Նոր Արեշի 35-րդ փողոց 1272, Երևան",0.0,NaN,NaN,NaN,...,1/3/2023 17:50,44.516050,40.154548,Sasuntsi Davit,1098,Erebuni,44.516040,40.154539,Nor_Butania,1/27/2023
55359,sold,18735943,61000.000000,NaN,NaN,Yerevan › Shengavit,0.0,NaN,NaN,NaN,...,1/3/2023 17:54,44.490789,40.148496,Shengavit,854,Shengavit,44.490781,40.148499,Verin_Shengavit,3/14/2023
56858,sold,18829370,5490.391814,NaN,NaN,"14-я улица Сари Таха 1, Ереван",0.0,NaN,NaN,NaN,...,NaN,44.528141,40.163625,Zoravar Andranik,2753,Erebuni,44.528150,40.163621,Sari_Tagh,1/27/2023


In [217]:
df = pd.concat([sale_historical,sale_actual], axis=0)

In [219]:
df.drop(columns=[ 'sold_date'], inplace=True)

In [220]:
df_clean = df.dropna().reset_index(drop=True)

In [224]:
df.to_csv(r'../database/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../database/yerevan_combined_clean/sale_clean_combined.csv', index=False)